In [ ]:
"""
Interaktywna mapa oszacowanego popytu na ladowanie EV - Polska.

Piec warstw:
1. Mapa ciepla - wszyscy kandydaci pod nowa inwestycje naraz, caly kraj
2. Top kandydaci korytarzowi (junction/mop/fuel_station sklasyfikowane
   jako korytarzowe) - wyfiltrowani wg NMS Top 1000
3. Top kandydaci docelowi (fuel_station/supermarket sklasyfikowane jako
   docelowe) - wyfiltrowani wg NMS Top 1000
4. Lokalni Liderzy Powiatowi - top 25 w goracych powiatach (diamentowe punkty)
5. Istniejaca infrastruktura jako kontekst - male, przygaszone znaczniki,
   wylaczone domyslnie
6. Potencjal powiatu (tlo) - suma wyniku kandydatow docelowych w powiecie
"""

import os
import json
import pandas as pd
import folium
from folium.plugins import HeatMap
import branca.colormap as cm

PLIK_SCORING = "../data/candidate_locations_ze_scoringiem.csv" if os.path.exists("../data/candidate_locations_ze_scoringiem.csv") else "candidate_locations_ze_scoringiem.csv"
PLIK_GEOJSON = "../data/powiaty_teryt.geojson" if os.path.exists("../data/powiaty_teryt.geojson") else "powiaty_teryt.geojson"
PLIK_WYJSCIOWY_MAPA = "../Mapa/index.html" if os.path.exists("../Mapa") else "index.html"

TOP_N_NUMEROWANE = 200        # ile z KRAJOWEGO topu NMS dostaje widoczny numer
RANGA_GORACY_POWIAT = 0.7     # od jakiej rangi powiat liczy sie jako "goracy"
LOKALNI_LIDERZY_N = 10        # ilu lokalnych liderow gwarantujemy w goracym powiecie

df = pd.read_csv(PLIK_SCORING, low_memory=False)
if "dedup_status" in df.columns:
    df = df[df["dedup_status"] == "unique"].copy()

KANDYDACI_NOWE = [
    "fuel_station", "junction", "mop", "supermarket",
]

ISTNIEJACE = ["eipa_station", "osm_charging_station"]

kandydaci = df[df["source_layer"].isin(KANDYDACI_NOWE)].copy()
istniejace = df[df["source_layer"].isin(ISTNIEJACE)].copy()

print(f"Kandydaci pod nowa inwestycje: {len(kandydaci)} "
      f"(korytarzowa={len(kandydaci[kandydaci['segment']=='korytarzowa'])}, "
      f"docelowa={len(kandydaci[kandydaci['segment']=='docelowa'])})")
print(f"Istniejaca infrastruktura (kontekst): {len(istniejace)}")

# ---------- liczymy dane choropleth WCZESNIEJ ----------
kandydaci["_teryt_4cyfry"] = kandydaci["teryt_powiat_geo"].astype(str).str.zfill(4)
suma_docelowa_powiat = (
    kandydaci[kandydaci["segment"] == "docelowa"]
    .groupby("_teryt_4cyfry")["wynik_scoringowy"]
    .sum()
    .reset_index()
)
suma_docelowa_powiat["ranga_percentylowa"] = suma_docelowa_powiat["wynik_scoringowy"].rank(pct=True)
gorace_powiaty = set(
    suma_docelowa_powiat.loc[suma_docelowa_powiat["ranga_percentylowa"] >= RANGA_GORACY_POWIAT, "_teryt_4cyfry"]
)
print(f"Powiaty o wysokim potencjale regionalnym (ranga >= {RANGA_GORACY_POWIAT}): {len(gorace_powiaty)}")

mapa = folium.Map(
    location=[52.237049, 21.017532], 
    zoom_start=6, 
    tiles=None
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, HERE, Garmin, USGS, NGA, EPA, USDA",
    name="Warstwy mapy",
    control=True
).add_to(mapa)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Reference/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Garmin, USGS",
    name="Nazwy miast i dróg",
    overlay=True,
    control=True
).add_to(mapa)


# =========================================================================
# WARSTWA 1: Mapa cieplna, tylko kandydaci
# =========================================================================
heat_data = kandydaci[["latitude", "longitude", "wynik_scoringowy"]].copy()
heat_data["waga"] = heat_data["wynik_scoringowy"].rank(pct=True)

heat_gradient = {0.2: "#313695", 0.4: "#74add1", 0.6: "#fee090", 0.8: "#f46d43", 1.0: "#a50026"}
colormap_heat = cm.LinearColormap(
    colors=["#313695", "#74add1", "#fee090", "#f46d43", "#a50026"],
    vmin=0, vmax=1,
    caption="Mapa cieplna - potencjal pod nowa stacje (ranga percentylowa)",
)

heat_layer = folium.FeatureGroup(name="Mapa cieplna - kandydaci pod nowa stacje", show=True)
HeatMap(
    heat_data[["latitude", "longitude", "waga"]].values.tolist(),
    radius=12, blur=15, max_zoom=10,
    gradient=heat_gradient,
).add_to(heat_layer)
heat_layer.add_to(mapa)
colormap_heat.add_to(mapa)

# =========================================================================
# WARSTWY 2-3: Top kandydaci, osobno per segment (filtracja NMS Top 1000)
# =========================================================================

# Wyciągamy podzbiór punktów NMS Top 1000 dla obu segmentów, by wyznaczyć dynamiczny zakres vmin/vmax
top_kor = kandydaci[(kandydaci["segment"] == "korytarzowa") & (kandydaci["czy_top1000_nms"] == True)]
top_dest = kandydaci[(kandydaci["segment"] == "docelowa") & (kandydaci["czy_top1000_nms"] == True)]

colormap_kor = cm.LinearColormap(
    colors=["#a1d99b", "#31a354", "#00441b"],
    vmin=top_kor["wynik_scoringowy"].min() if len(top_kor) > 0 else 0,
    vmax=top_kor["wynik_scoringowy"].max() if len(top_kor) > 0 else 1,
    caption="Wynik scoringowy (kWh/rok) - kandydaci korytarzowi (NMS Top 1000)",
)

# Nasycona, dobrze widoczna paleta różowa -> fioletowa bez bladdych odcieni
colormap_dest = cm.LinearColormap(
    colors=["#f768a1", "#dd1c77", "#980043", "#49006a"],
    vmin=top_dest["wynik_scoringowy"].min() if len(top_dest) > 0 else 0,
    vmax=top_dest["wynik_scoringowy"].max() if len(top_dest) > 0 else 1,
    caption="Wynik scoringowy (kWh/rok) - kandydaci docelowi (NMS Top 1000)",
)

segment_config = [
    ("korytarzowa", "Top kandydaci korytarzowi pod nowa stacje (NMS)", colormap_kor),
    ("docelowa", "Top kandydaci docelowi pod nowa stacje (NMS)", colormap_dest),
]

for seg_name, layer_label, colormap in segment_config:
    pula_segmentu = kandydaci[kandydaci["segment"] == seg_name]
    
    # ŚCISŁY FILTR TOP 1000 NMS
    top_krajowy = (
        pula_segmentu[pula_segmentu["czy_top1000_nms"] == True]
        .sort_values("pozycja_ranking_nms")
        .copy()
    )
    top_krajowy["pozycja"] = top_krajowy["pozycja_ranking_nms"]

    layer = folium.FeatureGroup(name=layer_label, show=True)
    max_wynik = top_krajowy["wynik_scoringowy"].max() if len(top_krajowy) > 0 else 1

    for _, row in top_krajowy.iterrows():
        pozycja = row["pozycja"]
        wynik = row["wynik_scoringowy"]
        energia_surowa = row.get("energia_kwh_rocznie_szacunek", 0)
        sesje = row.get("sesje_rocznie_szacunek", 0)
        pewnosc = row.get("skladowa_pewnosc_danych", row.get("pewnosc_danych_mnoznik", 1.0))
        percentyl = row.get("ranking_scoringowy_procentyl", float("nan"))
        percentyl_klasa = row.get("ranking_wewnatrz_klasy_procentyl", float("nan"))
        klasa_pow = row.get("klasa_powiatu", "n/a")
        promien = 4 + 10 * (wynik / max_wynik) ** 0.5
        hex_color = colormap(wynik)

        uwaga_zanizone = ""
        if row.get("dane_prawdopodobnie_zanizone", False):
            uwaga_zanizone = "<br><i>Uwaga: lokalna flota EV w tym powiecie jest prawdopodobnie zanizona w danych zrodlowych.</i>"

        popup_html = f"""
        <b>#{int(pozycja)} w rankingu NMS Top 1000 --- {row.get('name', 'brak nazwy') if pd.notna(row.get('name')) else 'brak nazwy'}</b><br>
        Typ: {row['source_layer']} (kandydat pod nowa stacje)<br>
        Powiat: {row.get('powiat_nazwa', 'n/a')}<br>
        Klasa powiatu: {klasa_pow}<br>
        Segment: {seg_name}<br>
        <hr>
        <b>Wynik scoringowy:</b> {wynik:,.0f} kWh/rok (percentyl: {percentyl:.0%})<br>
        Percentyl w klasie powiatu: {percentyl_klasa:.0%}<br>
        Mnoznik pewnosci danych: {pewnosc:.2f}<br>
        Surowy szacunek popytu (przed korekta): {energia_surowa:,.0f} kWh/rok<br>
        Sesje/rok: {sesje:,.0f}
        {uwaga_zanizone}
        <hr>
        Ruch (sam. osobowe/dobe): {row.get('traffic_primary_sam_osobowe', 0):,.0f}<br>
        Konkurencja aktywna, moc (2km): {row.get('existing_eipa_power_kw_active_2km', 0):.0f} kW<br>
        """

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=promien,
            color=hex_color,
            fill=True,
            fill_color=hex_color,
            fill_opacity=0.85,  # Zwiększone krycie dla lepszej widoczności
            weight=1.5,
            popup=folium.Popup(popup_html, max_width=320),
        ).add_to(layer)

        if pozycja <= TOP_N_NUMEROWANE:
            folium.Marker(
                location=[row["latitude"], row["longitude"]],
                icon=folium.DivIcon(html=f'''
                    <div style="
                        font-size: 9px; font-weight: bold; color: white;
                        background: rgba(0,0,0,0.7); border-radius: 50%;
                        width: 18px; height: 18px; text-align: center;
                        line-height: 18px; font-family: sans-serif;
                        transform: translate(-9px, -9px);">
                        {int(pozycja)}
                    </div>
                '''),
            ).add_to(layer)

    layer.add_to(mapa)
    colormap.add_to(mapa)

# =========================================================================
# WARSTWA 4: Lokalni Liderzy Powiatowi (Top 25 z gorących powiatów - diamenty)
# =========================================================================
pula_docelowa = kandydaci[kandydaci["segment"] == "docelowa"]

# Wykluczamy punkty, które zakwalifikowały się do NMS Top 1000
top_krajowy_ids = set(pula_docelowa[pula_docelowa["czy_top1000_nms"] == True]["location_id"])

kandydaci_z_goracych = pula_docelowa[pula_docelowa["_teryt_4cyfry"].isin(gorace_powiaty)]
lokalni_liderzy = (
    kandydaci_z_goracych.groupby("_teryt_4cyfry", group_keys=False)
    .apply(lambda g: g.nlargest(LOKALNI_LIDERZY_N, "wynik_scoringowy"))
)

# Filtrujemy tylko tych, ktorzy NIE zmiescili sie w przefiltrowanym NMS Top 1000
tylko_lokalni = lokalni_liderzy[~lokalni_liderzy["location_id"].isin(top_krajowy_ids)].copy()

liderzy_layer = folium.FeatureGroup(name="Lokalni Liderzy Powiatowi (Top 25)", show=True)
max_wynik_dest = pula_docelowa["wynik_scoringowy"].max()

for _, row in tylko_lokalni.iterrows():
    wynik = row["wynik_scoringowy"]
    energia_surowa = row.get("energia_kwh_rocznie_szacunek", 0)
    sesje = row.get("sesje_rocznie_szacunek", 0)
    pewnosc = row.get("skladowa_pewnosc_danych", row.get("pewnosc_danych_mnoznik", 1.0))
    percentyl = row.get("ranking_scoringowy_procentyl", float("nan"))
    percentyl_klasa = row.get("ranking_wewnatrz_klasy_procentyl", float("nan"))
    klasa_pow = row.get("klasa_powiatu", "n/a")
    promien = 4 + 10 * (wynik / max_wynik_dest) ** 0.5
    hex_color = colormap_dest(wynik)
    dim = max(8, int(promien * 1.8))

    popup_html = f"""
    <b>Lokalny lider powiatowy --- {row.get('name', 'brak nazwy') if pd.notna(row.get('name')) else 'brak nazwy'}</b><br>
    Typ: {row['source_layer']} (kandydat pod nowa stacje)<br>
    Powiat: {row.get('powiat_nazwa', 'n/a')}<br>
    Klasa powiatu: {klasa_pow}<br>
    Segment: docelowa<br>
    <hr>
    <b>Wynik scoringowy:</b> {wynik:,.0f} kWh/rok (percentyl: {percentyl:.0%})<br>
    Percentyl w klasie powiatu: {percentyl_klasa:.0%}<br>
    Mnoznik pewnosci danych: {pewnosc:.2f}<br>
    Surowy szacunek popytu: {energia_surowa:,.0f} kWh/rok<br>
    Sesje/rok: {sesje:,.0f}<br>
    <hr>
    Ruch (sam. osobowe/dobe): {row.get('traffic_primary_sam_osobowe', 0):,.0f}<br>
    Konkurencja aktywna, moc (2km): {row.get('existing_eipa_power_kw_active_2km', 0):.0f} kW<br>
    """

    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        icon=folium.DivIcon(html=f'''
            <div style="
                width: {dim}px; height: {dim}px;
                background-color: {hex_color};
                opacity: 0.85;
                border: 1px solid #49006a;
                transform: translate(-{dim//2}px, -{dim//2}px) rotate(45deg);">
            </div>
        '''),
        popup=folium.Popup(popup_html, max_width=320),
    ).add_to(liderzy_layer)

liderzy_layer.add_to(mapa)

# =========================================================================
# WARSTWA 5: Istniejaca infrastruktura, kontekst/walidacja
# =========================================================================
istniejace_layer = folium.FeatureGroup(name="Istniejaca infrastruktura (kontekst)", show=False)
for _, row in istniejace.iterrows():
    wynik = row["wynik_scoringowy"]
    percentyl = row.get("ranking_scoringowy_procentyl", float("nan"))
    popup_html = f"""
    <b>{row.get('name', 'brak nazwy') if pd.notna(row.get('name')) else 'brak nazwy'}</b><br>
    Typ: {row['source_layer']} (istniejaca stacja)<br>
    Operator: {row.get('operator', 'nieznany') if pd.notna(row.get('operator')) else 'nieznany'}<br>
    Powiat: {row.get('powiat_nazwa', 'n/a')}<br>
    Segment: {row.get('segment', 'n/a')}<br>
    <hr>
    <b>Jak model ocenia to miejsce:</b><br>
    Wynik scoringowy: {wynik:,.0f} kWh/rok (percentyl: {percentyl:.0%})<br>
    """
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=3,
        color="#555555",
        fill=True,
        fill_color="#7f8c8d",
        fill_opacity=0.5,
        weight=0.5,
        popup=folium.Popup(popup_html, max_width=280),
    ).add_to(istniejace_layer)
istniejace_layer.add_to(mapa)

# =========================================================================
# WARSTWA 6: Choropleth powiatow, potencjał pod stacje docelowe (tło)
# =========================================================================
if os.path.exists(PLIK_GEOJSON):
    with open(PLIK_GEOJSON, encoding="utf-8") as f:
        geo_powiaty = json.load(f)

    folium.Choropleth(
        geo_data=geo_powiaty,
        data=suma_docelowa_powiat,
        columns=["_teryt_4cyfry", "ranga_percentylowa"],
        key_on="feature.properties.JPT_KOD_JE",
        fill_color="YlOrRd",
        fill_opacity=0.6,
        line_opacity=0.3,
        line_color="white",
        nan_fill_color="#f0f0f0",
        legend_name="Potencjal powiatu pod NOWE stacje docelowe (ranga percentylowa)",
        name="Potencjal powiatu pod nowe stacje (tlo)",
        show=False,
    ).add_to(mapa)

folium.LayerControl(collapsed=False).add_to(mapa)

# =========================================================================
# PANEL INFORMACYJNY HTML (WERSJA Z OPISEM MODELU)
# =========================================================================
panel_html = f"""
<div id="panel-info" style="
    position: fixed; bottom: 20px; left: 20px; z-index: 9999;
    background: white; border: 2px solid #888; border-radius: 8px;
    box-shadow: 2px 2px 10px rgba(0,0,0,0.25); font-family: sans-serif;
    max-width: 360px;">
  <div onclick="
      var t = document.getElementById('panel-info-tresc');
      t.style.display = (t.style.display === 'none') ? 'block' : 'none';
    " style="
      cursor: pointer; padding: 9px 12px; font-weight: bold; font-size: 13px;
      background: #1a1a2e; color: white; border-radius: 6px 6px 0 0;
      display: flex; justify-content: space-between; align-items: center;">
    <span>ℹ️ O modelu i warstwach mapy</span>
    <span style="font-size: 10px; opacity: 0.8;">(kliknij)</span>
  </div>
  <div id="panel-info-tresc" style="display: block; padding: 12px; font-size: 12px; line-height: 1.45; max-height: 480px; overflow-y: auto;">

    <div style="background: #f8f9fa; padding: 8px; border-left: 3px solid #1a1a2e; margin-bottom: 10px;">
      <b>Jak liczony jest wynik?</b><br>
      Model szacuje roczne zapotrzebowanie na energię (kWh/rok) na podstawie:
      <ul style="margin: 4px 0 0 16px; padding: 0;">
        <li>Floty EV i populacji w powiecie,</li>
        <li>Średniego dobowego ruchu drogowego (GPR),</li>
        <li>Kary za obecną moc konkurencji (1–2 km),</li>
        <li>Wymogów Unijnych AFIR dla trasy TEN-T.</li>
      </ul>
    </div>

    <b>🔥 Mapa cieplna (kandydaci)</b><br>
    Całościowe zagęszczenie potencjału popytowego pod nowe stacje w całym kraju.<br><br>

    <b>🟢 Top korytarzowe (NMS 2 km)</b><br>
    Węzły, MOP-y i stacje przy trasach. Odrzuca punkty w buforze 100 m od stacji EIPA oraz stosuje filtr NMS = 2 km Numery 1–{TOP_N_NUMEROWANE} wskazują liderów w kraju.<br><br>

    <b>🟣 Top docelowe (NMS 2 km)</b><br>
    Supermarkety i stacje paliw w miastach. Filtrowanie analogiczne (100 m od EIPA, 2 km NMS). Numery 1–{TOP_N_NUMEROWANE} to czołówka krajowa.<br><br>

    <b>♦️ Lokalni Liderzy Powiatowi</b><br>
    Najlepsze miejsca w powiatach o wysokim potencjale, które nie trafiły do ścisłego Top NMS.<br><br>

    <b>⚪ Istniejąca infrastruktura (kontekst)</b><br>
    Działające ładowarki (rejestr EIPA / OSM) stanowiące odniesienie dla analizy konkurencji.<br><br>

    <b>🗺️ Potencjał powiatu (tło)</b><br>
    Skumulowany potencjał ładowania docelowego w skali całego powiatu.<br><br>

    <div style="border-top: 1px solid #eee; padding-top: 6px; color: #666; font-size: 10.5px;">
      <i>Uwaga: Wyniki mają charakter rankingowy i wskazują względną atrakcyjność lokalizacji.</i>
    </div>
  </div>
</div>
"""
mapa.get_root().html.add_child(folium.Element(panel_html))

mapa.save(PLIK_WYJSCIOWY_MAPA)
print(f"\nZapisano mape: {PLIK_WYJSCIOWY_MAPA}")
print(f"Liczba zaprezentowanych punktow Top NMS: {len(kandydaci[kandydaci['czy_top1000_nms'] == True])}")
print(f"Z widoczna numeracja (top {TOP_N_NUMEROWANE} z rankingu NMS w kazdym segmencie)")
print(f"Heatmapa (kandydaci): {len(kandydaci)}")
print(f"Lokalni liderzy (diamenty): {len(tylko_lokalni)}")
print(f"Istniejaca infrastruktura (kontekst): {len(istniejace)}")

Kandydaci pod nowa inwestycje: 15961 (korytarzowa=2436, docelowa=13525)
Istniejaca infrastruktura (kontekst): 6773
Powiaty o wysokim potencjale regionalnym (ranga >= 0.7): 115

Zapisano mape: ../Mapa/index.html
Liczba zaprezentowanych punktow Top NMS: 2000
Z widoczna numeracja (top 200 z rankingu NMS w kazdym segmencie)
Heatmapa (kandydaci): 15961
Lokalni liderzy (diamenty): 681
Istniejaca infrastruktura (kontekst): 6773
